<a href="https://colab.research.google.com/github/markkk20g/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print('Menu Detector!')

Menu Detector!


In [12]:
# -----------------
# Import Libraries
# -----------------
from google.colab import drive

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset_path:', DATASET_PATH)

# CAPITAL letters are global standart in this case

CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert',
    'cheesecake': 'dessert',
    'kebab': 'kebab',
    'pilaf': 'pilaf'
    # dataset kop Categoriyalarga ajratib ketishi mumkin, ex: choco-cake, cheesecake ==> Dessert
    # label grouping | class consolidation
    # ex2: apple, mongo ==> fruit
}

CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf'] # ozimizi class larni tashkillashtirdi
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)} # enumerate() har bir classga raqamlarni qoshib tashkillashtir beradi
NUM_CLASSES = len(CLASSES)

print(NUM_CLASSES)
print(CLASS_TO_IDX)

# new class starts from here
transform = transforms.Compose([
    transforms.Resize((224, 224)), # size w,h  | har bir image har hil size ga ega, biz hamma image ga bir hil size berdik!
    transforms.ToTensor(), # tensorga ogirib beradi , yani raqamlarga! FLOAT korinishiga 0.0 ~ 1.0 orasida boladi | 255 == 1.0
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]) #rasmlarni normal holatga keltiradi,
])
# Compose methodi bir qiladigon ishlarni ketma ket tashkillashtir beradi
# fine-tune == unknown for now!

# image -> H, W, CH ===> CH, H, W | Channel === RGB
# Normalize formula == pixel = (pixel - mean) / std  |||||||

Dataset_path: /content/drive/MyDrive/food101_dataset
5
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}


=============================================

In [5]:
# --------------------------
# Custom Dataset Class
# --------------------------

class FoodDataset(Dataset):
  def __init__(self, images, labels, transform=None):
    self.images = images
    self.labels = labels
    self.transform = transform

  def __len__(self):
    print('images_length:', len(self.images)) # data qancha? Biz keyin uni train va test ga ajratamiz
    return len(self.images)

  def __getitem__(self, idx): # asosiy ish!
    img_path = self.images[idx] # index orqali rasmlarni obkeladi
    print('image_path', img_path)
    label = self.labels[idx] # label ni index orqali aniqledi | ex: index 4 == pilaf
    print('label', label)
    try:
      image = Image.open(img_path).convert('RGB') # rasmni oqish va RGB ga ogirish
    except (UnidentifiedImageError, OSError): # file bn muammo bolsa (empty or wrong), terminalga print qil va keyingisiga ot!
      print(f'Skipping broken image: {img_path}')
      return self.__getitem__((idx + 1) % len(self.images))
    if self.transform:
      image = self.transform(image)
    return image, label

In [6]:
# --------------------------
# Gather and Split Data
# --------------------------


all_images = [] # oddiy empty list
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items(): # data iteration
    class_path = os.path.join(DATASET_PATH, original_class) # class path yaratyapmiz, yani DATASET_PATH ga original_class ni qoshberadi
    # before '/content/drive/MyDrive/food101_dataset' , now '/content/drive/MyDrive/food101_dataset/hamburger
    print('class_path:', class_path)
    if not os.path.exists(class_path): # class bolmasa warning terminalga chop qiladi va keyingisiga otib ketadi
      print(f'Warning: {class_path} not found')
      continue
    for img in os.listdir(class_path): # class_path dagi img larni oqib chiqadi
      if img.endswith(('.jpg', '.jpeg', '.png')): # .txt, .docx file lar bor, ularni oqimedi! faqat img formatdagilani oqidi
        full_path = os.path.join(class_path, img) # /content/drive/MyDrive/food101_dataset/hamburger/100057.jpg
        all_images.append((full_path, CLASS_TO_IDX[mapped_class])) # shu path ni va uni qaysi classga TEGISHLI ekanini korsatadi!!!

np.random.shuffle(all_images) # shuffle data | rasmlar random holatda ketma ketlikda joylashadi!! aralash-quralash
split = int(0.8 * len(all_images)) # split data
train_data = all_images[:split] # train data
val_data = all_images[split:] # test data

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, bl = dataset[0]

class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
images_length: 3326
3326
image_path /content/drive/MyDrive/food101_dataset/hamburger/3768429.jpg
label 0


================== 184

In [7]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform) # train un | data larni transform qilish MUMKIN deb oldik!
val_dataset = FoodDataset(val_images, val_labels, transform=transform) # train dan keyin Test qilish un

In [8]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2) # data ni obkelib uni ketma ketligini belgilab olyappiz
# batch_size=32 === bittada 32ta rasmni ai modelga train un beryappiz | num_workers=2 === backgroundda 2ta ishchi, similar to THREAD or Parallel loading for speed

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2) # test un | shuffle=False ketma ketlikda datani beradi

images_length: 3326
images_length: 3326


In [9]:
# PRE-TRAINED MODEL
model = mobilenet_v2(weights='IMAGENET1K_V1') # pretrained model | lightweight = ishlashi yengil | CNN == Convolutional Neural Network | 1000 class | 1M img trained
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
# eng muhim!!! shu modelga ozimizi class larimizi beryappiz | hamma featurelarni ishlatishni aytib ohiriga classimizi qoshdik |
# FINE-TUNING == 1000 classdan faqat biz bergan 5ta class boyicha train amalga oshir, qoganini ishlatma deb oldik
# Fine-tuning = pre-trained modelni olib Ozimizga MOSLASHtirish | 1000ga sohasi bolsa, bizga eng kereligini ovolib shunga togirlemiz, qoganini ishlatmimiz
# BACKBONE == mobilenet modelni miyasini qanday bolsa shunday ishlatyappiz
# Model layer freeze == modelni layer / bolimlarini muzlatib qoyish mumkin!

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 231MB/s]


In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)
model = model.to(device)

device cuda


In [16]:
# Loss function & Optimizer

criterion = nn.CrossEntropyLoss() # Loss Function = multi-class classification un CrossEntropyLoss kop ishlatiladi, model predict qilishda 70% burger, 30% pilaf deb ajratberadi
optimizer = optim.Adam(model.parameters(), lr=0.001) # train jarayonida bizga Parametrlarni (WEIGHT) update qilib turadi har safar
torch.backends.cudnn.benchmark = True # Benchmark Setting == Trick! aynan GPU or torch bn ishlashda shu GPU ga mos keladigon eng tez algorithmni ozi tanlab beradi, Performance optimization!